In [1]:
# IMPORTS

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix)

import joblib

In [2]:
# GPU CHECK
print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU detected:")
    for gpu in gpus:
        print("  ", gpu)
else:
    print("WARNING: No GPU detected.")

TensorFlow version: 2.20.0
GPU detected:
   PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
   PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')


In [16]:
# CONFIG

DATASET_FOLDER = Path("/kaggle/input/datasets/gaspard3129/prepared-dataset/prepared_dataset")

OUTPUT_FOLDER = Path("/kaggle/working")

MODEL_PATH = OUTPUT_FOLDER / "gesture_cnn.keras"
SCALERS_PATH = OUTPUT_FOLDER / "position_scalers.joblib"
LABEL_ENCODER_PATH = OUTPUT_FOLDER / "label_encoder.joblib"

RESULTS_FOLDER = OUTPUT_FOLDER / "cnn_experiments"
RESULTS_FOLDER.mkdir(exist_ok=True)

NUMBER_OF_TEST_PARTICIPANTS = 2
NUMBER_OF_CV_FOLDS = 5

RANDOM_STATE = 20

In [17]:
# FEATURES

FEATURE_COLUMNS = [
    "Time", "LeftPosX", "LeftPosY", "LeftPosZ", "LeftRotX", "LeftRotY",
    "LeftRotZ", "LeftRotW", "RightPosX", "RightPosY", "RightPosZ",
    "RightRotX", "RightRotY", "RightRotZ", "RightRotW"
]

POSITION_FEATURES = [
    "LeftPosX", "LeftPosY", "LeftPosZ", "RightPosX", "RightPosY", "RightPosZ"
]

POSITION_INDICES = [FEATURE_COLUMNS.index(feature) for feature in POSITION_FEATURES]

In [18]:
def load_dataset():

    X = []
    y = []
    participants = []

    participant_folders = sorted(DATASET_FOLDER.glob("Participant_*"))

    participant_count = len(participant_folders)
    sample_count = 0

    for participant in participant_folders:

        participant_name = participant.name

        gesture_folders = sorted(p for p in participant.iterdir() if p.is_dir())

        for gesture_folder in gesture_folders:

            gesture_name = gesture_folder.name
            csv_files = sorted(gesture_folder.glob("*.csv"))

            for csv_file in csv_files:

                df = pd.read_csv(csv_file)
                df = df[FEATURE_COLUMNS]
                sample = df.to_numpy(dtype=np.float32)

                X.append(sample)
                y.append(gesture_name)
                participants.append(participant_name)

                sample_count += 1

    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    participants = np.array(participants)

    print()
    print("Dataset loaded")
    print("--------------------")
    print("Participants :", participant_count)
    print("Samples      :", sample_count)
    print("Classes      :", len(np.unique(y)))
    print("Sample shape :", X.shape[1:])
    print("Dataset shape:", X.shape)

    return X, y, participants

In [19]:
def encode_labels(y):

    label_encoder = LabelEncoder()

    y_encoded = label_encoder.fit_transform(y)

    return y_encoded, label_encoder

In [20]:
def split_dataset(X, y, participants):

    unique_participants = np.unique(participants)

    rng = np.random.default_rng(RANDOM_STATE)

    shuffled_participants = unique_participants.copy()
    rng.shuffle(shuffled_participants)

    test_participants = np.sort(shuffled_participants[:NUMBER_OF_TEST_PARTICIPANTS])
    development_participants = np.sort(shuffled_participants[NUMBER_OF_TEST_PARTICIPANTS:])

    test_mask = np.isin(participants, test_participants)
    development_mask = np.isin(participants, development_participants)

    X_development = X[development_mask]
    y_development = y[development_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    print()
    print("Participant split")
    print("--------------------")

    print("Test participants:")
    for participant in test_participants:
        print(" ", participant)

    print()
    print("Development participants:")
    for participant in development_participants:
        print(" ", participant)

    print()
    print("Development samples :", len(X_development))
    print("Test samples        :", len(X_test))

    return (X_development, y_development, X_test, y_test, test_participants)

In [21]:
def normalize_positions(X_train, X_other):

    X_train = X_train.copy()
    X_other = X_other.copy()

    scalers = {}

    for feature_index in POSITION_INDICES:

        feature_name = FEATURE_COLUMNS[feature_index]
        scaler = StandardScaler()
        train_values = (X_train[:, :, feature_index].reshape(-1, 1))
        scaler.fit(train_values)

        X_train[:, :, feature_index] = (scaler.transform(train_values)
            .reshape(X_train.shape[0], X_train.shape[1]))

        other_values = (X_other[:, :, feature_index].reshape(-1, 1))

        X_other[:, :, feature_index] = (scaler.transform(other_values)
            .reshape(X_other.shape[0], X_other.shape[1]))

        scalers[feature_name] = scaler

    return X_train, X_other, scalers

In [22]:
# CNN MODEL

def create_cnn_model(input_shape, num_classes, filters=(64, 64, 128), kernel_size=3,
    dropout_rates=(0.3, 0.3, 0.4), l2_strength=1e-4, learning_rate=1e-3):

    model = keras.Sequential()
    model.add(layers.Input(shape=input_shape))
    model.add(layers.BatchNormalization())

    for i, number_of_filters in enumerate(filters):

        model.add(layers.Conv1D(filters=number_of_filters, kernel_size=kernel_size,
                activation="relu", padding="same", kernel_regularizer=l2(l2_strength)))

        model.add(layers.BatchNormalization())

        if i < len(dropout_rates):

            model.add(layers.Dropout(dropout_rates[i]))

    model.add(layers.GlobalAveragePooling1D())
    model.add(layers.Dense(128, activation="relu", kernel_regularizer=l2(l2_strength)))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(num_classes, activation="softmax"))

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy", 
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")])

    return model

In [23]:
def train_model(architecture, X_train, y_train, X_val, y_val, num_classes):

    model = create_cnn_model(input_shape=X_train.shape[1:], num_classes=num_classes,
        filters=architecture)

    start_time = time.perf_counter()

    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
        epochs=50, batch_size=32, verbose=0)

    training_time = (time.perf_counter() - start_time)
    val_probabilities = model.predict(X_val, verbose=0)
    val_predictions = np.argmax(val_probabilities, axis=1)

    accuracy = accuracy_score(y_val, val_predictions)
    balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)

    f1 = f1_score(y_val, val_predictions, average="macro", zero_division=0)

    return (model, history, accuracy, balanced_accuracy, f1, training_time)

In [24]:
# LOAD DATA

print("\nLoading dataset...\n")

X, y, participants = load_dataset()
y, label_encoder = encode_labels(y)

num_classes = len(label_encoder.classes_)

(X_development, y_development, X_test, y_test, test_participants) = split_dataset(
    X, y, participants)


Loading dataset...


Dataset loaded
--------------------
Participants : 14
Samples      : 1540
Classes      : 22
Sample shape : (50, 15)
Dataset shape: (1540, 50, 15)

Participant split
--------------------
Test participants:
  Participant_10
  Participant_14

Development participants:
  Participant_01
  Participant_02
  Participant_03
  Participant_04
  Participant_05
  Participant_06
  Participant_07
  Participant_08
  Participant_09
  Participant_11
  Participant_12
  Participant_13

Development samples : 1320
Test samples        : 220


In [25]:
# CNN ARCHITECTURE SEARCH

architectures = [
    (32, 32), (64, 64), (64, 64, 128), (128, 128), (128, 128, 256), (64, 128, 256)
]

kf = KFold(n_splits=NUMBER_OF_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
results = []

best_architecture = None
best_cv_f1 = -1

total_experiments = (len(architectures) * NUMBER_OF_CV_FOLDS)
experiment_number = 0

for architecture in architectures:

    print()
    print("========================================")
    print("Architecture:", architecture)
    print("========================================")

    fold_accuracies = []
    fold_f1_scores = []
    fold_training_times = []

    for fold, (train_indices, val_indices) in enumerate(kf.split(X_development), start=1):

        experiment_number += 1
        print(f"Fold {fold}/{NUMBER_OF_CV_FOLDS}")

        X_train = X_development[train_indices]
        X_val = X_development[val_indices]

        y_train = y_development[train_indices]
        y_val = y_development[val_indices]

        # Normalize using training fold only
        (X_train, X_val, _) = normalize_positions(X_train, X_val)

        (model, history, accuracy, balanced_accuracy, f1, training_time) = train_model(
            architecture, X_train, y_train, X_val, y_val, num_classes)

        fold_accuracies.append(accuracy)
        fold_f1_scores.append(f1)
        fold_training_times.append(training_time)

        print(f"    Accuracy : {accuracy:.4f}")
        print(f"    F1       : {f1:.4f}")
        print(f"    Time     : {training_time:.2f}s")

        # Free the model before the next fold
        del model
        tf.keras.backend.clear_session()

    # architecture results

    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)
    mean_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)
    mean_training_time = np.mean(fold_training_times)

    results.append({
        "architecture": str(architecture),
        "accuracy_mean": mean_accuracy,
        "accuracy_std": std_accuracy,
        "f1_mean": mean_f1,
        "f1_std": std_f1,
        "training_time_mean":mean_training_time
    })

    print()
    print(
        f"Mean accuracy : "
        f"{mean_accuracy:.4f} "
        f"+/- {std_accuracy:.4f}"
    )

    print(
        f"Mean F1       : "
        f"{mean_f1:.4f} "
        f"+/- {std_f1:.4f}"
    )

    # Select using macro-F1
    if mean_f1 > best_cv_f1:

        best_cv_f1 = mean_f1
        best_architecture = architecture


# Save results

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="f1_mean",ascending=False)
results_df.to_csv(RESULTS_FOLDER / "cnn_results.csv",index=False)

print()
print("========================================")
print("CNN ARCHITECTURE RESULTS")
print("========================================")

print(results_df)

print()
print("Best architecture:")
print(best_architecture)

print(
    f"Best CV macro-F1: "
    f"{best_cv_f1:.4f}"
)


Architecture: (32, 32)
Fold 1/5


I0000 00:00:1786816710.293499      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786816710.296303      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786816719.096149     801 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    Accuracy : 0.9735
    F1       : 0.9743
    Time     : 18.95s
Fold 2/5
    Accuracy : 0.9848
    F1       : 0.9821
    Time     : 16.85s
Fold 3/5
    Accuracy : 0.9621
    F1       : 0.9621
    Time     : 16.86s
Fold 4/5
    Accuracy : 0.9583
    F1       : 0.9573
    Time     : 16.95s
Fold 5/5
    Accuracy : 0.9735
    F1       : 0.9740
    Time     : 17.17s

Mean accuracy : 0.9705 +/- 0.0094
Mean F1       : 0.9699 +/- 0.0090

Architecture: (64, 64)
Fold 1/5
    Accuracy : 0.9848
    F1       : 0.9853
    Time     : 17.87s
Fold 2/5
    Accuracy : 0.9848
    F1       : 0.9853
    Time     : 17.37s
Fold 3/5
    Accuracy : 0.9848
    F1       : 0.9850
    Time     : 17.24s
Fold 4/5
    Accuracy : 0.9697
    F1       : 0.9690
    Time     : 17.29s
Fold 5/5
    Accuracy : 0.9621
    F1       : 0.9668
    Time     : 17.45s

Mean accuracy : 0.9773 +/- 0.0096
Mean F1       : 0.9783 +/- 0.0085

Architecture: (64, 64, 128)
Fold 1/5
    Accuracy : 0.9811
    F1       : 0.9819
    Time     : 

In [26]:
# FINAL MODEL

print()
print("========================================")
print("TRAINING FINAL MODEL")
print("========================================")

# Normalize using ALL development data
X_train_final, X_test_final, final_scalers = (normalize_positions(
        X_development, X_test))

final_model = create_cnn_model(input_shape=X_train_final.shape[1:],
    num_classes=num_classes, filters=best_architecture
)

start_time = time.perf_counter()

final_history = final_model.fit(X_train_final, y_development,
    epochs=50, batch_size=32, verbose=0)

final_training_time = (time.perf_counter() - start_time)

print(
    f"Training time: "
    f"{final_training_time:.2f} seconds"
)


# FINAL TEST

print()
print("========================================")
print("FINAL TEST EVALUATION")
print("========================================")

test_probabilities = final_model.predict(X_test_final, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)
test_accuracy = accuracy_score(y_test, test_predictions)
test_balanced_accuracy = balanced_accuracy_score(y_test, test_predictions)

test_precision = precision_score(y_test, test_predictions, average="macro", zero_division=0)
test_recall = recall_score(y_test, test_predictions, average="macro", zero_division=0)
test_f1 = f1_score(y_test, test_predictions, average="macro", zero_division=0)

print()
print(
    "Test participants:",
    ", ".join(test_participants)
)

print(
    f"Test accuracy          : "
    f"{test_accuracy:.4f}"
)

print(
    f"Test balanced accuracy : "
    f"{test_balanced_accuracy:.4f}"
)

print(
    f"Test precision         : "
    f"{test_precision:.4f}"
)

print(
    f"Test recall            : "
    f"{test_recall:.4f}"
)

print(
    f"Test F1                : "
    f"{test_f1:.4f}"
)

# Classification report
print()
print("Classification report")
print("--------------------")

report = classification_report(y_test, test_predictions,
    target_names=label_encoder.classes_, zero_division=0
)
print(report)

with open(RESULTS_FOLDER / "test_classification_report.txt", "w") as file:

    file.write(report)

# Final test results

final_results = pd.DataFrame([{
    "best_architecture": str(best_architecture),
    "test_participants": ", ".join(test_participants),
    "accuracy": test_accuracy,
    "balanced_accuracy": test_balanced_accuracy,
    "precision_macro": test_precision,
    "recall_macro": test_recall,
    "f1_macro": test_f1,
    "training_time_seconds": final_training_time
}])

final_results.to_csv(RESULTS_FOLDER / "final_test_results.csv", index=False)


TRAINING FINAL MODEL
Training time: 23.01 seconds

FINAL TEST EVALUATION

Test participants: Participant_10, Participant_14
Test accuracy          : 0.8091
Test balanced accuracy : 0.8091
Test precision         : 0.8488
Test recall            : 0.8091
Test F1                : 0.7880

Classification report
--------------------
               precision    recall  f1-score   support

          Add       1.00      1.00      1.00        10
          Bar       1.00      1.00      1.00        10
Brewed Coffee       1.00      1.00      1.00        10
   Cappuccino       1.00      0.60      0.75        10
     Espresso       0.77      1.00      0.87        10
Hot Chocolate       1.00      1.00      1.00        10
         Iced       1.00      1.00      1.00        10
        Large       1.00      1.00      1.00        10
        Latte       1.00      0.50      0.67        10
       Medium       0.77      1.00      0.87        10
         Milk       1.00      1.00      1.00        10
        Or

In [27]:
# SAVE MODEL

print()
print("Saving model...")

final_model.save(MODEL_PATH)
joblib.dump(final_scalers, SCALERS_PATH)
joblib.dump(label_encoder, LABEL_ENCODER_PATH)

print()
print("Files saved:")
print("--------------------")

print(MODEL_PATH)
print(SCALERS_PATH)
print(LABEL_ENCODER_PATH)


Saving model...

Files saved:
--------------------
/kaggle/working/gesture_cnn.keras
/kaggle/working/position_scalers.joblib
/kaggle/working/label_encoder.joblib
